1. 환경 설치

In [2]:
!pip install -r requirements.txt

1. 임포트

In [3]:
import ace_lib as ace
from dataset_to_region import dataset_to_region

2. 세션 생성

In [4]:
s = ace.start_session()

Complete biometrics authentication and press any key to continue: 
https://api.worldquantbrain.com/authentication/persona?inquiry=inq_jB6iw4fRfHMwEbyc2ERqXccrxBf4



3. 데이터셋 조회

3-1. 단일 데이터셋 조회

In [5]:
dataset_id = "analyst10"
region = dataset_to_region(s, dataset_id, ace_module=ace)
print(f"{dataset_id} → {region}")

INFO:dataset_to_region:Cache invalid or expired. Reloading datasets...
INFO:dataset_to_region:Fetching datasets from Brain API...
INFO:dataset_to_region:Successfully fetched 208 datasets
INFO:dataset_to_region:Dataset 'analyst10' mapped to region 'USA'


analyst10 → USA


3-2. 여러 데이터셋 조회

In [6]:
dataset_ids = ["analyst10", "analyst11", "news97"]

for dataset_id in dataset_ids:
    region = dataset_to_region(s, dataset_id)
    print(f"{dataset_id} → {region}")

INFO:dataset_to_region:Using cached dataset information
INFO:dataset_to_region:Dataset 'analyst10' mapped to region 'USA'
INFO:dataset_to_region:Using cached dataset information
INFO:dataset_to_region:Dataset 'analyst11' mapped to region 'USA'
INFO:dataset_to_region:Using cached dataset information
INFO:dataset_to_region:Dataset 'news97' mapped to region 'USA'


analyst10 → USA
analyst11 → USA
news97 → USA


4. 시뮬레이션

In [18]:
import json
with open('json.json') as fl:
    alpha_list=json.load(fl)
result = ace.simulate_alpha_list_multi(s, alpha_list[:10])

2026-01-04 19:27:58,643 - ace - WARNING - List of alphas too short, single concurrent simulations will be used instead of multisimulations
100%|██████████| 4/4 [02:31<00:00, 37.82s/it]


In [19]:
result

[{'alpha_id': 'akJ7d5mO',
  'simulate_data': {'type': 'REGULAR',
   'settings': {'instrumentType': 'EQUITY',
    'region': 'USA',
    'universe': 'TOP3000',
    'delay': 0,
    'decay': 0,
    'neutralization': 'INDUSTRY',
    'truncation': 0.08,
    'pasteurization': 'OFF',
    'testPeriod': 'P1Y',
    'unitHandling': 'VERIFY',
    'nanHandling': 'ON',
    'maxTrade': 'OFF',
    'language': 'FASTEXPR',
    'visualization': False},
   'regular': 'Rank(Ts_Rank(close, 10))'},
  'is_stats':        pnl  bookSize  longCount  shortCount  turnover  returns  drawdown  \
  0 -4299912  20000000       1550        1561    0.7129  -0.0431    0.5181   
  
       margin  sharpe  fitness   startDate  \
  0 -0.000121   -0.56    -0.14  2013-01-20   
  
                              investabilityConstrained  \
  0  {'pnl': -5578474, 'bookSize': 20000000, 'longC...   
  
                                       riskNeutralized  alpha_id  
  0  {'pnl': -5423101, 'bookSize': 20000000, 'longC...  akJ7d5mO  ,
 

In [22]:
import pandas as pd
import json
import numpy as np

def clean_nan_fields(obj):
    """NaN 값을 가진 필드를 제거"""
    if isinstance(obj, dict):
        # NaN인 키-값 쌍 제거
        return {
            k: clean_nan_fields(v) 
            for k, v in obj.items() 
            if not (isinstance(v, float) and np.isnan(v))
        }
    elif isinstance(obj, list):
        return [clean_nan_fields(item) for item in obj]
    else:
        return obj

def convert_dataframes(obj):
    """DataFrame을 dict로 변환"""
    if isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient='records')
    elif isinstance(obj, dict):
        return {k: convert_dataframes(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_dataframes(item) for item in obj]
    else:
        return obj

# 사용 예시
serializable_result = convert_dataframes(result)
cleaned_result = clean_nan_fields(serializable_result)

with open('result.json', 'w', encoding='utf-8') as jsonfile:
    json.dump(cleaned_result, jsonfile, indent=4, ensure_ascii=False)